# Weight demo

This notebook shows how to connect and drive a kern weight scale from Python.

## 1. Initialisation

In [204]:
import time
import csv
import threading
import numpy as np


from enderscope import WeightScale
from enderscope import Stage, ScanPatterns
from enderscope import Magnetometer, MagnetometerLogger
from pathlib import Path
from datetime import datetime

In [206]:
scale = WeightScale('COM11', baud_rate=9600)   # balance
ender = Stage('COM12', 115200)                   # plateau déplacement (X, Y, Z)
mag = Magnetometer('COM3', baud_rate=115200)    # magnétomètre (5 x MLX90393)

print(scale)
print(ender)
print(mag)

## 2. Positionnement / Force

### Homing et positionnement de la balance

<div style="background-color:#ffe0e0; border: 3px solid #d90000; border-radius: 8px; padding: 14px;">
<h2 style="color:#d90000; margin-top:0;">⚠️ ATTENTION ⚠️</h2>
<p style="font-size:16px; color:#7a0000;"><strong>RETIRER L'INDENTEUR avant de lancer le homing ci-dessous !</strong><br></p>
</div>

In [211]:
ender.home()
ender.set_speed(6000)
ender.move_relative(0, 0, 60)
ender.finish_moves()

### Point de départ

Coordonnées du premier point de mesure à définit selon la manip (X, Y, Z)

In [215]:
x_depart = 100
y_depart = 150
z_depart = 48.5

ender.move_absolute(x_depart, y_depart, z_depart)
ender.finish_moves()

### Tare de la balance

In [243]:
scale.perform_tare()

Empty answer from scale
Empty answer from scale
Empty answer from scale
Empty answer from scale


### Fonction d'indentation

In [221]:
def apply_force(max_force=100, speed=60, max_retries=10, retry_delay=0.1):
    force = 0
    ender.set_speed(speed)

    while force < max_force:
        # Lecture AVANT de bouger
        reading = None
        for _ in range(max_retries):
            reading = scale.get_instant_reading()
            if reading is not None:
                break
            time.sleep(retry_delay)

        if reading is None:
            raise RuntimeError(
                f"Balance injoignable apres {max_retries} tentatives -- "
                f"arret par securite, aucun mouvement supplementaire "
                f"(derniere force connue : {force}g)"
            )
        force = reading

        if force < max_force:
            if force < 0.8 * max_force:
                ender.move_relative(0, 0, -0.1)
            else:
                ender.set_speed(int(speed / 2))
                ender.move_relative(0, 0, -0.05)
            ender.finish_moves()

    time.sleep(0.5)
    reading = scale.get_instant_reading()
    if reading is not None:
        force = reading
    end_pos = ender.get_position()
    #print (f"position: {end_pos} - force: {force}")
    return force, end_pos

## 3. Magnétomètre

### Lecture brute

In [225]:
values = mag.get_reading()
print(values)

[-4617.60009765625, 2.4000000953674316, -518.8480224609375, -3686.400146484375, -960.0000610351562, -1870.176025390625, 976.800048828125, 3271.2001953125, 1521.696044921875, 268.8000183105469, -2436.0, 247.80799865722656, -314.4000244140625, -1716.0001220703125, -143.2639923095703]


### Baseline

Moyenne de quelques lectures au repos pour corriger les mesures.

In [227]:
baseline = mag.acquire_baseline(num_samples=5)
print('Baseline magnétomètre :', baseline)

Baseline magnétomètre : [-4.61616016e+03  1.92000008e+00 -5.25043188e+02 -3.68400024e+03
 -9.57600037e+02 -1.86243201e+03  9.78240027e+02  3.26880005e+03
  1.52479355e+03  2.67360022e+02 -2.43312012e+03  2.43161603e+02
 -3.14400024e+02 -1.71456011e+03 -1.37843201e+02]


## 4. Acquisition & enregistrement CSV

### Dossier de sortie

Chaque prise de mesure crée son propre dossier, nommé par la date/heure et le type de mesure
(ex: `20260918_143210_mesure_combinee`). 

In [232]:
def create_output_dir(measurement_type: str) -> Path:
    timestamp_str = datetime.now().strftime('%Y%m%d_%H%M%S')
    output_dir = Path(f'{timestamp_str}_{measurement_type}') 
    output_dir.mkdir(parents=True, exist_ok=True)
    return output_dir

### Plage de force

In [235]:
force_targets = np.arange(100, 700, 200)  
print(force_targets)

[100 300 500]


### Motif de scan

Grille de positions autour du point de départ.

In [238]:
origin = [x_depart, y_depart] # coordinates of first position
steps = [10,-10] # x and y shifts to next item
positions = ScanPatterns.snake(3,3) * steps + origin
#ScanPatterns.plot_path(positions, field=(0,0),title='snake scan')

### a) Boucle de scan — mesure combinée (position + force)

1. création du dossier de sortie horodaté pour cette mesure
2. démarrage du `MagnetometerLogger` (flux indépendant, `sensor.csv` / `sensor_post_baselines.csv`)
3. boucle de scan : indentation à chaque position, écriture de `states.csv` avec son propre timestamp
4. fin du scan

In [ ]:
output_dir = create_output_dir('mesure_combinee')
print('Dossier de sortie :', output_dir)

mag_logger = MagnetometerLogger(
    mag, baseline,
    sensor_path=output_dir / 'sensor.csv',
    corrected_path=output_dir / 'sensor_post_baselines.csv',
)
mag_logger.start()

try:
    with open(output_dir / 'states.csv', 'w', newline='') as f_states:
        writer_states = csv.writer(f_states)

        for scan_pos in positions:
            ender.move_position(scan_pos)
            ender.finish_moves()
            start_pos = ender.get_position()

            scale.perform_tare()  # remise à zéro avant d'indenter à cette position

            for target_force in force_targets:
                force, end_pos = apply_force(target_force)
                print(f'z: {end_pos[2]} - force: {force}g (cible: {target_force}g)')

                timestamp = time.time()
                x, y, z = end_pos
                writer_states.writerow([timestamp, x, y, z, force])

            ender.set_speed(6000)
            ender.move_position(start_pos)
            ender.finish_moves()
finally:
    mag_logger.stop()
    mag_logger.join()

Dossier de sortie : 20260923_190136_mesure_combinee
Empty answer from scale
z: 44.9 - force: 101.0g (cible: 100g)
z: 41.55 - force: 300.4g (cible: 300g)
z: 39.15 - force: 501.5g (cible: 500g)
Empty answer from scale
z: 45.0 - force: 101.9g (cible: 100g)
z: 41.75 - force: 301.4g (cible: 300g)
z: 39.35 - force: 500.8g (cible: 500g)
Empty answer from scale
Empty answer from scale
z: 45.0 - force: 100.4g (cible: 100g)
z: 41.75 - force: 303.1g (cible: 300g)
z: 39.45 - force: 501.3g (cible: 500g)
Empty answer from scale
Empty answer from scale
z: 45.65 - force: 100.3g (cible: 100g)
z: 42.55 - force: 301.3g (cible: 300g)
z: 40.2 - force: 503.8g (cible: 500g)
Empty answer from scale
Empty answer from scale
z: 45.65 - force: 101.7g (cible: 100g)
z: 42.6 - force: 301.6g (cible: 300g)
z: 40.3 - force: 500.5g (cible: 500g)
Empty answer from scale
Empty answer from scale


### b) Boucle de scan — mesure position seule
TODO

### c) Boucle de scan — mesure force seule
TODO

## Annexe : Test indentation — montee incrementale de la force jusqu'a une masse cible

Objectif : verifier si le silicone se dechire en approchant une force elevee (seuil souhaité 3000 g).


In [ ]:
FORCE_TARGET_MAX_G = 3000   # masse cible finale
FORCE_STEP_G = 200          # increment entre deux paliers
MAX_DEPTH_MM = 30           # profondeur relative max pour securite -- au-dela, arret auto

force_steps = list(range(FORCE_STEP_G, FORCE_TARGET_MAX_G + 1, FORCE_STEP_G))
print(f"Paliers prevus (g) : {force_steps}")

scale.perform_tare()
start_pos = ender.get_position()
z_depart = start_pos[2]

previous_depth = 0.0

for palier_g in force_steps:
    print(f"\n--- Palier cible : {palier_g} g ---")

    force, end_pos = apply_force(max_force=palier_g, speed=60)
    z = end_pos[2]
    profondeur = z_depart - z  # deplacement relatif depuis le debut du test (mm)

    print(f"Force atteinte : {force:.1f} g (cible {palier_g} g) -- profondeur relative : {profondeur:.2f} mm")

    # Securité 1 : profondeur totale excessive
    if profondeur > MAX_DEPTH_MM:
        print(f"ATTENTION : profondeur ({profondeur:.2f} mm) au-dela du seuil de securite ({MAX_DEPTH_MM} mm).")
        print("Arret automatique -- le silicone a peut-etre cede (perte de resistance).")
        break

    # Securite 2 : saut anormal de profondeur par rapport au palier precedent
    # (signe possible indentation excessive)
    delta = profondeur - previous_depth
    if previous_depth > 0 and delta > 3 * (MAX_DEPTH_MM / len(force_steps)):
        print(f"ATTENTION : saut de profondeur inhabituel entre ce palier et le precedent "
              f"({delta:.2f} mm) -- possible perte de resistance du silicone.")
    previous_depth = profondeur

    reponse = input(
        f"Palier {palier_g} g atteint. Inspection du silicone, "
        f"puis appuyer sur Entree pour continuer, ou taper 'stop' pour arreter ici : "
    )
    if reponse.strip().lower() == 'stop':
        print("Arret demande par l'utilisateur.")
        break

print("\nRemontee en position de depart...")
ender.set_speed(6000)
ender.move_position(start_pos)
ender.finish_moves()
print("Test termine.")
